# 05 Кросс-таблица_U&A_ЦА #

In [ ]:
import pandas as pd
from brandpulse_api.calc import BrandpulseCalc
from brandpulse_api.upload import BrandpulseUpload

# имя файла с данными
DB_FILE = 'brandpulse3.duckdb'

# создаем объект для расчетов
bp = BrandpulseCalc(db_file=DB_FILE)

# создаем объект для загрузки
upload = BrandpulseUpload(db_file=DB_FILE)

Исследуем доступные проекты

In [ ]:
# вывод всех проектов
#bp.find_projects()

# поиск по подстроке в имени проекта
# bp.find_projects(project_name='желуд')

# поиск по двум возможным вариантам имени проекта
# bp.find_projects(project_name='желуд|майон')

# поиск по подстроке в английском имени проекта
#bp.find_projects(project_name_eng='gastro')

# поиск по системному имени категории
#bp.find_projects(category_sys_name='JEL')

# поиск по имени категории
#bp.find_projects(category_name='желуд')

# поиск по английскому имени категории
#bp.find_projects(category_name_eng='gastro')

# поиск по дате начала волны 
#bp.find_projects(wave_begin='2025-09-01')

# поиск по дате конца волны 
#bp.find_projects(wave_end='2025-09-28')

# поиск по имени волны 
#bp.find_projects(wave_name="Сентябрь'25")

# поиск проекта по нескольким параметрам
bp.find_projects(project_name='продуктов и готовой еды', wave_name="Январь'25|Февраль'25")


Пример передачи найденных ид проектов и их описаний для загрузки

In [ ]:
# ищем проекты, как выше, но сохряняем их в переменную
prj = bp.find_projects(project_name='продуктов и готовой еды', wave_name="Январь'25|Февраль'25")

# выводим найденные имена для наглядности
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    print(prj['project_name'])
    
# сохраняем описание проекта для вывода далее в шапке файла
description = prj['description'].iloc[0]
print(description)

# отбираем все значения в столбце project_id (на всякий случай сохраняем только уникальные значения)
prj_ids = prj['project_id'].unique().tolist()
prj_ids

In [ ]:
# загружаем найденные проекты
#upload.upload_projects(project_list=prj_ids)

Справочно исследуем свойства

In [ ]:
# поиск по описанию
# bp.find_property(text='Пол')

# поиск по имени
# bp.find_property(sys_name='SEX')

# поиск по комбинации имени и описания
# bp.find_property(text='Возраст', sys_name='SEX')

Справочно исследуем значения свойств

In [ ]:
# поиск по описанию
#bp.find_property_options(text='Покупали кофе')

# поиск по имени
#bp.find_property_options(sys_name='ORDERED_FOOD_TO')

# поиск по комбинации имени и описания
#bp.find_property_options(text='кофе', sys_name='COFFEE')

# поиск по имени свойства (символ ^ добавлен для поиска имен, начинающихся с этой строки)
# bp.find_property_options(property_sys_name='^AIM_RIT_BUK18')

Задаем параметры расчета

In [ ]:
project_ids = [1738676442715462656, 3468058483162416448]

# задаем параметры целевой группы (используя sys_name из базы)
demo = [
    {"property_name": "SEX", 
     "expression": "='MALE'"},
    {"property_name": "AGE_COUNT2", 
     "expression": " BETWEEN 'A25' AND 'A44' "}
]

Рассчитаем universe

In [ ]:
# пример расчета по атрибутам и их значениям
properties = [
    ('FREQUSE2_', ['TWO_THR_TIMES_M', 'ONCE_A_WEEK', 'SEVERAL_TIMES_W', 'ONCE_A_MONTH', 'MORE_SELDOM', 'EVERY_DAY', 'NO_ANSWER']),
    ('AIM_RIT_BUK8', ['TO_AVOID', 'TOI_INDUL', 'SO_AS', 'SPEC_OCC', 'UNEXPECTE', 'REG_LUNCH', 'ORDE_F_ANO', 'OTHER']),
    ('IST_INF_RIT', ['NOTHING', 'REK_ROD', 'REK', 'REK_V_INT', 'REK_IZV']),
    ('FACT_VYB_RIT2', ['THE_PLACE', 'HIGH_GOODS', 'EASY_DELIV', 'PROMOTIONS_SALES', 'GREAT_RANGE', 'GOOD_REPUTATION', 'HIGH_SERVI', 'LOW_PRICE', 
                       'CONVENIENT_PAYME','CONVENIENT_INTER', 'FAMOUS_BRAND', 'ITEMS_ELSEW', 'NEW_PLACE', 'NONE_ABOVE', 'HIGH_PRICE']),
    ('CEN_SEG_RIT', ['SRED', 'BUDG', 'DOROG']),
    ('VOL_TIME2', ['MINUTES_1530', 'HALF_HOUR', 'LESS_THAN_15', 'HOURS_12', 'MORE_THAN_2']),
    ('AIM_RIT_BUK18', ['FOR_THE_DEL', 'LARG_PURCH', 'QUICKL_BUY', 'PURC_ABO', 'FIND_A', 'ORDE_F_ANO', 'OTHER']),
    ]

df1 = bp.calc_project_property_universe(project_ids=project_ids, demo=demo, properties=properties)
df1

Считаем размер сэмпла

In [ ]:
# добавляем расчет сэмпла
sample = bp.calc_sample(project_ids=project_ids, demo=demo)
sample

Считаем размер universe по всему профилю и по целевой группе

In [ ]:
# добавляем расчет юниверс и тотал юнивес
total_universe = bp.calc_universe(project_ids=project_ids, demo=None)

universe = bp.calc_universe(project_ids=project_ids, demo=demo)

print(total_universe)
print(universe)

Добавляем к рассчитанному universe расчет долей (col%)

In [ ]:
# добавляем расчет доли
df_tmp = df1.copy()
df_tmp['universe_col%'] = round(100 * df_tmp['universe'] / universe, 1)

# добавляем строку с итогами
total_row = pd.DataFrame({
    'property_sys_name': [''],
    'property_text': [''],
    'option_sys_name': [''],
    'option_text': ['total'],
    'universe': [universe],
    'universe_col%': [100]
})

# маскируем сэмпл меньше 5
sample_limit = 5
df_tmp['universe_col%'] = df_tmp['universe_col%'].mask(
    (df_tmp['sample'] < sample_limit) | (df_tmp['sample'].isnull()), 
    '~')
df_tmp['universe'] = df_tmp['universe'].mask(
    (df_tmp['sample'] < sample_limit) | (df_tmp['sample'].isnull()), 
    '~')


# убираем колонку сэмпла
df_tmp.drop(columns=['sample'], inplace=True)

result = pd.concat([total_row, df_tmp], 
                   ignore_index=True)
result

Сохраняем в excel с форматированием отчета

In [ ]:
# сохранение в эксель с форматированием
import os

subfolder = "excel"  # название подпапки
os.makedirs(subfolder, exist_ok=True)  # создаёт подпапку, если не существует

filepath = os.path.join(subfolder, "05_crosstab_ua_soc_dem.xlsx")

with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
    result.to_excel(writer, sheet_name='Расчет', startrow=8, startcol=1, index=False)
    
    # настройка листа экселя
    worksheet = writer.sheets['Расчет']
    
    # Устанавливаем ширину столбцов (учитываем startcol=1)
    worksheet.column_dimensions['B'].width = 20  
    worksheet.column_dimensions['C'].width = 20 
    worksheet.column_dimensions['D'].width = 20  
    worksheet.column_dimensions['E'].width = 20 
    worksheet.column_dimensions['F'].width = 20  
    worksheet.column_dimensions['J'].width = 20  
    
    worksheet['A1'] = 'База данных: Brandpulse 2024'
    worksheet['A2'] = f'Размер генеральной совокупности (тыс.): {total_universe}'
    worksheet['A3'] = "Целевая база: Население <Январь'25, Февраль'25> [BHT_Доставка продуктов и готовой еды, Россия 100k+, 14-64, лично пользовались за последние 6 месяцев]"
    worksheet['A4'] = f'Размер целевой базы (тыс.): {total_universe}'
    worksheet['A5'] = 'Целевая группа: М 25-44'
    worksheet['A6'] = f'Размер целевой группы (тыс.): {universe}     Выборка: {sample}'
    worksheet['A7'] = f'Размер (%): {round(100 * universe / total_universe, 1)}%'
    
    worksheet['B9'] = ''
    worksheet['C9'] = ''
    worksheet['D9'] = ''
    worksheet['E9'] = ''
    worksheet['G9'] = 'col%'